# Peatland Hyperspectral Data — Comprehensive EDA

**Dataset**: Salko et al. 2024. Hyperspectral characterization of vegetation in hemiboreal, boreal and Arctic peatlands (Finland & Estonia). Mendeley doi:10.17632/3866tj3w8v.1  
**Spectra**: 446 plots · 350–2500 nm · 2151 bands (1 nm step)  
**Goal**: Understand the structure of the data before modelling — distributions, spectral patterns, geographic gradients, PFT compositions, and inter-variable relationships.

---
### Notebook sections
1. Setup & data loading  
2. Dataset overview and quality audit  
3. Geographic distribution  
4. Target variable: Finnish peatland type  
5. Plant functional type (PFT) composition  
6. Tree basal area  
7. Raw spectral overview  
8. Mean spectra by peatland type  
9. Mean spectra by site / latitude gradient  
10. Spectral band statistics (variance, range)  
11. Per-band discriminability (ANOVA F-score & mutual information)  
12. Correlation: PFT cover vs. reflectance  
13. Dimensionality: PCA  
14. PCA biplots (type, latitude, site)  
15. PCA loading vectors  
16. Spectral indices (NDVI, NDWI, CAI)  
17. SWIR quality class audit  
18. Summary & key findings  

## 1. Setup & Data Loading

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from matplotlib.colors import Normalize
import matplotlib.cm as cm
from scipy.stats import pearsonr, f_oneway
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.feature_selection import f_classif, mutual_info_classif

# ── Aesthetics ────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi"       : 130,
    "font.family"      : "sans-serif",
    "font.size"        : 10,
    "axes.spines.top"  : False,
    "axes.spines.right": False,
    "axes.grid"        : True,
    "grid.alpha"       : 0.25,
    "grid.linestyle"   : "--",
})

# ── Column constants ──────────────────────────────────────────────────────────
TARGET_COL = "Finnish_peatland_type"
SITE_COL   = "Site"
LAT_COL    = "Coordinate_y"
LON_COL    = "Coordinate_x"
COUNTRY_COL= "Country"

PFT_COLS = [
    "PFT_bare_peat", "PFT_brown_mosses", "PFT_graminoids", "PFT_herbaceous",
    "PFT_lichens", "PFT_litter", "PFT_other_mosses", "PFT_sphagnum_mosses",
    "PFT_woody_stemmed", "PFT_water", "PFT_unidentified",
]
PFT_LABELS = [c.replace("PFT_", "").replace("_", " ") for c in PFT_COLS]

BA_COLS = [
    "BA_Pine_living", "BA_Pine_dead",
    "BA_Spruce_living", "BA_Spruce_dead",
    "BA_Deciduous_living", "BA_Deciduous_dead",
]

# Noisy water-absorption bands (nm) — excluded from analyses unless stated
NOISY_BANDS = set(range(1330, 1550)) | set(range(1761, 2025)) | set(range(2311, 2501))
NOISY_COLS  = {f"wl{b}" for b in NOISY_BANDS}

print("Libraries loaded successfully.")

In [ ]:
# ── CHANGE THIS PATH to point to your CSV ────────────────────────────────────
CSV_FILE = r"C:\Users\aggarwm1\Videos\multi-disciplinary_Hyperspectral\data\Hyperspectral\Airborne_data\Reflectance_spectra_of_peatland_vegetation_Finland_Estonia_smoothed.csv"
# If using the raw file: CSV_FILE = r"...raw.csv"

df = pd.read_csv(CSV_FILE, sep=",", low_memory=False)

# Ensure coordinate columns are numeric
df[LAT_COL] = pd.to_numeric(df[LAT_COL], errors="coerce")
df[LON_COL] = pd.to_numeric(df[LON_COL], errors="coerce")

# Helper functions
def get_spec_cols(exclude_noisy=True):
    cols = [c for c in df.columns if c.startswith("wl")]
    return [c for c in cols if c not in NOISY_COLS] if exclude_noisy else cols

def wl_array(cols):
    return np.array([int(c[2:]) for c in cols])

def impute(X):
    """Column-mean imputation for any residual NaNs."""
    X = X.copy()
    means = np.nanmean(X, axis=0)
    nans  = np.where(np.isnan(X))
    X[nans] = np.take(means, nans[1])
    return X

spec_cols_clean = get_spec_cols(exclude_noisy=True)
spec_cols_all   = get_spec_cols(exclude_noisy=False)
wls             = wl_array(spec_cols_clean)
wls_all         = wl_array(spec_cols_all)

print(f"Rows × Columns : {df.shape}")
print(f"All spec bands : {len(spec_cols_all)}")
print(f"Clean bands    : {len(spec_cols_clean)}  (after excluding {len(NOISY_BANDS)} noisy nm)")
print(f"Wavelength range (clean): {wls.min()}–{wls.max()} nm")

## 2. Dataset Overview & Quality Audit

In [ ]:
# Basic counts
print("=" * 55)
print("DATASET OVERVIEW")
print("=" * 55)
print(f"Total plots    : {len(df)}")
print(f"Countries      : {df[COUNTRY_COL].value_counts().to_dict()}")
print(f"Unique sites   : {df[SITE_COL].nunique()}")
print(f"Plots per site :")
print(df[SITE_COL].value_counts().rename_axis("Site").rename("n_plots").to_frame().to_string())
print(f"\nDate range     : {df['Date'].min()} – {df['Date'].max()}")
print(f"\nLatitude range : {df[LAT_COL].min():.4f} – {df[LAT_COL].max():.4f} °N")
print(f"Longitude range: {df[LON_COL].min():.4f} – {df[LON_COL].max():.4f} °E")

In [ ]:
# Missing-value audit (non-spectral columns only)
meta_cols = ([TARGET_COL, SITE_COL, COUNTRY_COL, LAT_COL, LON_COL, "SWIR_class"] 
             + PFT_COLS + BA_COLS)
missing = df[meta_cols].isna().sum()
missing = missing[missing > 0]

if len(missing) == 0:
    print("No missing values in metadata columns.")
else:
    print("Missing values in metadata columns:")
    print(missing.to_string())

# Check spectral NaN rate
X_all = df[spec_cols_clean].values.astype(float)
nan_pct = np.isnan(X_all).mean() * 100
print(f"\nSpectral NaN rate (clean bands): {nan_pct:.2f}%")

In [ ]:
# SWIR quality class distribution
if "SWIR_class" in df.columns:
    fig, ax = plt.subplots(figsize=(6, 3))
    swir_counts = df["SWIR_class"].value_counts().sort_index()
    bars = ax.bar(swir_counts.index.astype(str), swir_counts.values,
                  color=["#2E7D32", "#FBC02D", "#C62828"][:len(swir_counts)],
                  edgecolor="white", width=0.6)
    ax.bar_label(bars, padding=3)
    ax.set_xlabel("SWIR quality class")
    ax.set_ylabel("Number of plots")
    ax.set_title("SWIR data quality class distribution\n(affects 2025–2310 nm region)")
    plt.tight_layout()
    plt.show()
    print(swir_counts)

## 3. Geographic Distribution

In [ ]:
sites   = sorted(df[SITE_COL].unique())
n_sites = len(sites)
site_cmap = plt.cm.get_cmap("tab20", n_sites)
site2c    = {s: site_cmap(i) for i, s in enumerate(sites)}

fi  = df.dropna(subset=[TARGET_COL])  # Finnish plots with type label
ee  = df[df[COUNTRY_COL] == "EE"]    # Estonian plots

types  = sorted(fi[TARGET_COL].unique())
t_cmap = plt.cm.get_cmap("Set2", len(types))
t2c    = {t: t_cmap(i) for i, t in enumerate(types)}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Panel A: by site ──
ax = axes[0]
for site in sites:
    sub = df[df[SITE_COL] == site]
    ax.scatter(sub[LON_COL], sub[LAT_COL], s=22, alpha=0.8,
               color=site2c[site], label=site, zorder=3)
ax.set_xlabel("Longitude (°E)"); ax.set_ylabel("Latitude (°N)")
ax.set_title("A: All plots coloured by site")
ax.legend(fontsize=6.5, ncol=2, loc="lower right",
          framealpha=0.9, edgecolor="#ccc")

# ── Panel B: by peatland type ──
ax = axes[1]
ax.scatter(ee[LON_COL], ee[LAT_COL], s=22, alpha=0.4,
           color="#888", marker="^", label="Estonia (no type label)", zorder=2)
for t in types:
    sub = fi[fi[TARGET_COL] == t]
    ax.scatter(sub[LON_COL], sub[LAT_COL], s=22, alpha=0.85,
               color=t2c[t], label=t, zorder=3)
ax.set_xlabel("Longitude (°E)"); ax.set_ylabel("Latitude (°N)")
ax.set_title("B: Finnish plots coloured by peatland type")
ax.legend(fontsize=6.5, ncol=2, loc="lower right",
          framealpha=0.9, edgecolor="#ccc")

plt.suptitle("Geographic distribution of 446 peatland plots (Finland & Estonia)",
             fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

print(f"\nLatitude span  : {df[LAT_COL].max() - df[LAT_COL].min():.2f}° " 
      f"(≈ {(df[LAT_COL].max() - df[LAT_COL].min()) * 111:.0f} km)")

## 4. Target Variable: Finnish Peatland Type

In [ ]:
counts = fi[TARGET_COL].value_counts()
n_fi   = len(fi)
n_ee   = len(ee)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# ── Horizontal bar chart ──
ax = axes[0]
colors = [t2c[t] for t in counts.index]
bars   = ax.barh(counts.index, counts.values, color=colors,
                 edgecolor="white", height=0.7)
ax.bar_label(bars, labels=[f"{v} ({v/n_fi*100:.0f}%)" for v in counts.values],
             padding=4, fontsize=8)
ax.set_xlabel("Number of plots")
ax.set_title(f"Peatland type distribution\n(Finnish plots, n={n_fi})")
ax.invert_yaxis()
ax.set_xlim(0, counts.max() * 1.35)

# ── Country split ──
ax = axes[1]
ax.pie([n_fi, n_ee], labels=[f"Finland\n(n={n_fi})", f"Estonia\n(n={n_ee})"],
       colors=["#3B7DBE", "#E07B39"], autopct="%1.0f%%",
       startangle=90, wedgeprops={"edgecolor": "white", "linewidth": 2})
ax.set_title("Plots by country")

plt.tight_layout()
plt.show()

print("\nClass counts:")
print(counts.to_string())
print(f"\nClass imbalance ratio (max/min): {counts.max()/counts.min():.1f}×")

In [ ]:
# Plots per site × type heatmap — reveals which types are at which sites
cross = fi.groupby([SITE_COL, TARGET_COL]).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(max(10, len(cross.columns)*1.1), max(4, len(cross)*0.55)))
im = ax.imshow(cross.values, aspect="auto", cmap="YlOrRd")
ax.set_xticks(range(len(cross.columns)))
ax.set_xticklabels(cross.columns, rotation=40, ha="right", fontsize=8)
ax.set_yticks(range(len(cross.index)))
ax.set_yticklabels(cross.index, fontsize=8)
for i in range(len(cross.index)):
    for j in range(len(cross.columns)):
        v = cross.values[i, j]
        if v > 0:
            ax.text(j, i, str(v), ha="center", va="center", fontsize=8,
                    color="white" if v > cross.values.max()*0.6 else "black")
plt.colorbar(im, ax=ax, label="n plots")
ax.set_title("Number of plots per Site × Peatland type\n(important for leave-one-site-out CV later)")
plt.tight_layout()
plt.show()

## 5. Plant Functional Type (PFT) Composition

In [ ]:
# ── Overall PFT distribution ──
pft_means  = df[PFT_COLS].mean()
pft_medians= df[PFT_COLS].median()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Sorted mean cover
ax = axes[0]
order = pft_means.sort_values(ascending=False)
bars  = ax.bar(range(len(order)), order.values,
               color=plt.cm.Paired(np.linspace(0, 1, len(order))),
               edgecolor="white")
ax.set_xticks(range(len(order)))
ax.set_xticklabels([l.replace("PFT_","").replace("_"," ") for l in order.index],
                   rotation=40, ha="right", fontsize=8)
ax.set_ylabel("Mean fractional cover (0–1)")
ax.set_title("Mean PFT cover across all 446 plots")

# Boxplots
ax = axes[1]
bp = ax.boxplot([df[c].dropna() for c in PFT_COLS],
                patch_artist=True, vert=True,
                medianprops={"color": "black", "lw": 1.5})
for patch, color in zip(bp["boxes"], plt.cm.Paired(np.linspace(0, 1, len(PFT_COLS)))):
    patch.set_facecolor(color)
ax.set_xticks(range(1, len(PFT_COLS)+1))
ax.set_xticklabels(PFT_LABELS, rotation=40, ha="right", fontsize=8)
ax.set_ylabel("Fractional cover (0–1)")
ax.set_title("PFT cover distributions (all plots)")

plt.tight_layout()
plt.show()

In [ ]:
# ── PFT composition by peatland type (stacked bar) ──
grp = fi.groupby(TARGET_COL)[PFT_COLS].mean()
pft_colors = plt.cm.Paired(np.linspace(0, 1, len(PFT_COLS)))

fig, ax = plt.subplots(figsize=(12, 5))
bottom = np.zeros(len(grp))
for j, (col, lbl) in enumerate(zip(PFT_COLS, PFT_LABELS)):
    vals = grp[col].fillna(0).values
    ax.bar(grp.index, vals, bottom=bottom, label=lbl,
           color=pft_colors[j], edgecolor="white", linewidth=0.4)
    bottom += vals

ax.set_ylabel("Mean fractional cover")
ax.set_ylim(0, 1.12)
ax.set_title("Mean PFT composition by peatland type")
ax.legend(fontsize=7.5, bbox_to_anchor=(1.01, 1), loc="upper left", ncol=1)
plt.xticks(rotation=35, ha="right", fontsize=8)
plt.tight_layout()
plt.show()

print("\nMean PFT cover by peatland type:")
print(grp.round(3).to_string())

In [ ]:
# ── PFT correlation matrix ──
pft_data = df[PFT_COLS].dropna(how="all")
corr_mat = pft_data.corr()

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(corr_mat.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(PFT_COLS)))
ax.set_yticks(range(len(PFT_COLS)))
ax.set_xticklabels(PFT_LABELS, rotation=40, ha="right", fontsize=8)
ax.set_yticklabels(PFT_LABELS, fontsize=8)
for i in range(len(PFT_COLS)):
    for j in range(len(PFT_COLS)):
        v = corr_mat.values[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=6.5,
                color="white" if abs(v) > 0.6 else "black")
plt.colorbar(im, ax=ax, label="Pearson r")
ax.set_title("PFT fractional cover — correlation matrix")
plt.tight_layout()
plt.show()

## 6. Tree Basal Area

In [ ]:
ba = df.copy()
ba["BA_total_living"]  = (ba["BA_Pine_living"].fillna(0)
                          + ba["BA_Spruce_living"].fillna(0)
                          + ba["BA_Deciduous_living"].fillna(0))
ba["BA_total_dead"]    = (ba["BA_Pine_dead"].fillna(0)
                          + ba["BA_Spruce_dead"].fillna(0)
                          + ba["BA_Deciduous_dead"].fillna(0))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Boxplot by peatland type
ax = axes[0]
ba_fi = ba.dropna(subset=[TARGET_COL])
type_order = ba_fi.groupby(TARGET_COL)["BA_total_living"].median().sort_values(ascending=False).index
data_by_type = [ba_fi[ba_fi[TARGET_COL]==t]["BA_total_living"].dropna() for t in type_order]
bp = ax.boxplot(data_by_type, patch_artist=True, vert=True,
                medianprops={"color": "black", "lw": 1.5})
for patch, t in zip(bp["boxes"], type_order):
    patch.set_facecolor(t2c.get(t, "#aaa"))
ax.set_xticks(range(1, len(type_order)+1))
ax.set_xticklabels(type_order, rotation=35, ha="right", fontsize=7.5)
ax.set_ylabel("Total living basal area (m²/ha)")
ax.set_title("Living basal area by peatland type")

# Species breakdown (stacked bar, means per type)
ax = axes[1]
sp_cols  = ["BA_Pine_living", "BA_Spruce_living", "BA_Deciduous_living"]
sp_labels= ["Pine", "Spruce", "Deciduous"]
sp_colors= ["#4CAF50", "#2196F3", "#FF9800"]
sp_grp   = ba_fi.groupby(TARGET_COL)[sp_cols].mean().reindex(type_order)
bot = np.zeros(len(type_order))
for col, lbl, col_c in zip(sp_cols, sp_labels, sp_colors):
    vals = sp_grp[col].fillna(0).values
    ax.bar(range(len(type_order)), vals, bottom=bot, label=lbl,
           color=col_c, edgecolor="white", linewidth=0.5)
    bot += vals
ax.set_xticks(range(len(type_order)))
ax.set_xticklabels(type_order, rotation=35, ha="right", fontsize=7.5)
ax.set_ylabel("Mean basal area (m²/ha)")
ax.set_title("Species composition of living basal area")
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 7. Raw Spectral Overview

In [ ]:
# Individual spectra sample (first 80 plots, semi-transparent)
X_all  = df[spec_cols_all].values.astype(float)
X_clean= df[spec_cols_clean].values.astype(float)

fig, ax = plt.subplots(figsize=(14, 5))

sample_idx = np.random.RandomState(0).choice(len(df), size=min(80, len(df)), replace=False)
for i in sample_idx:
    ax.plot(wls_all, X_all[i], color="#3B7DBE", alpha=0.07, lw=0.6)

# Global mean
global_mean = np.nanmean(X_all, axis=0)
ax.plot(wls_all, global_mean, color="#C62828", lw=1.8, label="Global mean", zorder=5)

# Shade noisy regions
for r, label in [((1330, 1549), "water abs."), ((1761, 2024), "water abs."), ((2311, 2500), "water abs.")]:
    ax.axvspan(r[0], r[1], color="gray", alpha=0.18, zorder=2)

# Region labels
for nm, lbl in [(530, "VIS"), (1060, "NIR"), (1650, "SWIR-1"), (2175, "SWIR-2")]:
    ax.text(nm, ax.get_ylim()[1]*0.02 if ax.get_ylim()[1] > 0 else 0.02,
            lbl, ha="center", fontsize=9, color="#444", style="italic")

ax.axvline(700, color="k", lw=0.7, linestyle="--", alpha=0.5, label="Red-edge (~700 nm)")
ax.set_xlabel("Wavelength (nm)")
ax.set_ylabel("Reflectance factor")
ax.set_title("Sample of 80 individual spectra with global mean (gray zones = excluded noisy bands)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Global reflectance distribution statistics
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Per-band mean and std across all plots
ax = axes[0]
band_mean = np.nanmean(X_clean, axis=0)
band_std  = np.nanstd(X_clean,  axis=0)
ax.plot(wls, band_mean, color="#3B7DBE", lw=1.2, label="Mean")
ax.fill_between(wls, band_mean - band_std, band_mean + band_std,
                color="#3B7DBE", alpha=0.2, label="±1 SD")
ax.fill_between(wls,
                np.nanpercentile(X_clean, 10, axis=0),
                np.nanpercentile(X_clean, 90, axis=0),
                color="#3B7DBE", alpha=0.08, label="10–90th pct")
ax.set_xlabel("Wavelength (nm)"); ax.set_ylabel("Reflectance factor")
ax.set_title("Global spectral statistics (all 446 plots)")
ax.legend(fontsize=8)

# Coefficient of variation per band
ax = axes[1]
cv = band_std / (band_mean + 1e-9)
ax.plot(wls, cv, color="#E07B39", lw=1.0)
ax.fill_between(wls, 0, cv, color="#E07B39", alpha=0.25)
ax.set_xlabel("Wavelength (nm)")
ax.set_ylabel("Coefficient of variation (SD/mean)")
ax.set_title("Band-wise coefficient of variation\n(high CV = more inter-plot variability)")
ax.axvline(700, color="k", lw=0.7, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

## 8. Mean Spectra by Peatland Type

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

for t in types:
    sub = fi[fi[TARGET_COL] == t][spec_cols_clean].values.astype(float)
    mu  = np.nanmean(sub, axis=0)
    sd  = np.nanstd(sub, axis=0)
    c   = t2c[t]
    ax.plot(wls, mu, color=c, lw=1.4, label=f"{t} (n={len(sub)})")
    ax.fill_between(wls, mu - sd, mu + sd, color=c, alpha=0.1)

ax.axvline(700,  color="k", lw=0.6, linestyle="--", alpha=0.4)
ax.axvline(800,  color="k", lw=0.6, linestyle=":",  alpha=0.4)
ax.text(700,  ax.get_ylim()[1]*0.97, "red-edge", fontsize=7.5,
        va="top", ha="right", rotation=90, color="gray")

ax.set_xlabel("Wavelength (nm)"); ax.set_ylabel("Mean reflectance factor")
ax.set_title("Mean (±1 SD) spectra by Finnish peatland type")
ax.legend(fontsize=7, ncol=3, loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
# Zoom into the red-edge (690–800 nm) — most visually distinct region
red_edge_mask = (wls >= 690) & (wls <= 810)
wls_re = wls[red_edge_mask]

fig, ax = plt.subplots(figsize=(9, 4))
for t in types:
    sub = fi[fi[TARGET_COL] == t][spec_cols_clean].values.astype(float)
    mu  = np.nanmean(sub, axis=0)[red_edge_mask]
    sd  = np.nanstd(sub, axis=0)[red_edge_mask]
    ax.plot(wls_re, mu, color=t2c[t], lw=1.8, label=t)
    ax.fill_between(wls_re, mu-sd, mu+sd, color=t2c[t], alpha=0.12)
ax.set_xlabel("Wavelength (nm)"); ax.set_ylabel("Reflectance factor")
ax.set_title("Red-edge detail (690–810 nm) — key discriminating region")
ax.legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.show()

## 9. Mean Spectra by Site / Latitude Gradient

In [ ]:
site_lats = df.groupby(SITE_COL)[LAT_COL].mean()
sites_sorted = site_lats.sort_values().index.tolist()  # south → north
lat_min, lat_max = site_lats.min(), site_lats.max()
lat_norm  = Normalize(lat_min, lat_max)
lat_cmap  = cm.RdYlGn_r  # warm = south, cool = north

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# ── Panel A: mean spectra coloured by latitude ──
ax = axes[0]
for site in sites_sorted:
    sub = df[df[SITE_COL] == site][spec_cols_clean].values.astype(float)
    mu  = np.nanmean(sub, axis=0)
    lat = site_lats[site]
    ax.plot(wls, mu, color=lat_cmap(lat_norm(lat)), lw=1.1,
            label=f"{site} ({lat:.2f}°N)")

sm = cm.ScalarMappable(cmap=lat_cmap, norm=lat_norm)
sm.set_array([])
plt.colorbar(sm, ax=ax, label="Mean latitude (°N)")
ax.set_xlabel("Wavelength (nm)"); ax.set_ylabel("Reflectance factor")
ax.set_title("Mean spectra per site, coloured by latitude")
ax.legend(fontsize=5.5, ncol=2, loc="upper right")

# ── Panel B: NIR reflectance vs latitude ──
ax = axes[1]
nir_col = "wl800"
if nir_col in df.columns:
    ax.scatter(df[LAT_COL], df[nir_col].astype(float),
               c=df[SITE_COL].astype("category").cat.codes,
               cmap="tab20", s=12, alpha=0.6)
    # Overlay site means
    for site in sites_sorted:
        sub = df[df[SITE_COL] == site]
        ax.scatter(sub[LAT_COL].mean(), sub[nir_col].astype(float).mean(),
                   s=90, color=lat_cmap(lat_norm(site_lats[site])),
                   edgecolors="black", lw=0.8, zorder=5,
                   marker="D", label=site)
    ax.set_xlabel("Latitude (°N)")
    ax.set_ylabel("NIR reflectance (800 nm)")
    ax.set_title("NIR reflectance vs. latitude\n(dots = plots, diamonds = site means)")

plt.suptitle("North–south spectral gradient across 13 sites", fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

## 10. Band-wise Statistics (Variance, Range, Skewness)

In [ ]:
from scipy.stats import skew

X_c   = impute(X_clean)
b_var  = np.var(X_c, axis=0)
b_range= np.max(X_c, axis=0) - np.min(X_c, axis=0)
b_skew = np.array([skew(X_c[:, j]) for j in range(X_c.shape[1])])

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

for ax, vals, label, color in zip(
        axes,
        [b_var, b_range, b_skew],
        ["Variance", "Range (max−min)", "Skewness"],
        ["#3B7DBE", "#E07B39", "#8E44AD"]):
    ax.fill_between(wls, vals, color=color, alpha=0.4, lw=0)
    ax.plot(wls, vals, color=color, lw=0.8)
    ax.axhline(0, color="k", lw=0.4)
    ax.set_ylabel(label, fontsize=9)

axes[-1].set_xlabel("Wavelength (nm)")
axes[0].set_title("Per-band spectral statistics across all 446 plots")
plt.tight_layout()
plt.show()

## 11. Per-Band Discriminability: ANOVA F-score & Mutual Information

In [ ]:
fi_idx = fi.index
X_fi   = impute(fi[spec_cols_clean].values.astype(float))
y_fi   = LabelEncoder().fit_transform(fi[TARGET_COL])

f_scores, _  = f_classif(X_fi, y_fi)
mi_scores    = mutual_info_classif(X_fi, y_fi, random_state=42)

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

for ax, vals, label, color in zip(
        axes,
        [f_scores, mi_scores],
        ["ANOVA F-score (higher = more discriminative between types)",
         "Mutual information (bits)"],
        ["#3B7DBE", "#E07B39"]):
    ax.plot(wls, vals, color=color, lw=0.8)
    ax.fill_between(wls, 0, vals, color=color, alpha=0.25)
    ax.set_ylabel(label, fontsize=9)

# Annotate well-known regions
for nm, lbl in [(680, "Chl red"), (720, "Red-edge"), (970, "H₂O"), (1150, "NIR shoulder")]:
    if wls.min() < nm < wls.max():
        axes[0].axvline(nm, color="gray", lw=0.8, linestyle="--", alpha=0.6)
        axes[0].text(nm+5, axes[0].get_ylim()[1]*0.9, lbl,
                     fontsize=7, va="top", rotation=90, color="gray")

axes[-1].set_xlabel("Wavelength (nm)")
axes[0].set_title("Per-band discriminability for Finnish_peatland_type classification")
plt.tight_layout()
plt.show()

# Top bands
top10_f  = wls[np.argsort(f_scores)[::-1][:10]]
top10_mi = wls[np.argsort(mi_scores)[::-1][:10]]
print(f"Top 10 most discriminative wavelengths (ANOVA F):    {top10_f} nm")
print(f"Top 10 most discriminative wavelengths (Mutual info): {top10_mi} nm")

## 12. PFT Cover vs. Reflectance: Pearson Correlations

In [ ]:
X_imp = impute(df[spec_cols_clean].values.astype(float))

pft_cmap = plt.cm.get_cmap("tab10", len(PFT_COLS))

fig, axes = plt.subplots(len(PFT_COLS), 1,
                         figsize=(14, 2.2 * len(PFT_COLS)), sharex=True)
for i, (col, lbl, ax) in enumerate(zip(PFT_COLS, PFT_LABELS, axes)):
    pft_vals = df[col].values.astype(float)
    valid    = ~np.isnan(pft_vals)
    r_vals   = np.array([
        pearsonr(pft_vals[valid], X_imp[valid, j])[0]
        for j in range(X_imp.shape[1])
    ])
    ax.fill_between(wls, r_vals, color=pft_cmap(i), alpha=0.45, lw=0)
    ax.plot(wls, r_vals, color=pft_cmap(i), lw=0.7)
    ax.axhline(0, color="k", lw=0.4)
    ax.set_ylim(-1, 1)
    ax.set_ylabel(lbl, fontsize=7.5)
    # Mark r = ±0.5
    ax.axhline( 0.5, color="gray", lw=0.4, linestyle=":")
    ax.axhline(-0.5, color="gray", lw=0.4, linestyle=":")

axes[-1].set_xlabel("Wavelength (nm)")
axes[0].set_title("Pearson r: each PFT fractional cover vs. reflectance by wavelength\n"
                  "(dotted lines = r = ±0.5)")
plt.tight_layout()
plt.show()

## 13. Dimensionality Reduction: PCA

In [ ]:
X_sc = StandardScaler().fit_transform(X_imp)
pca  = PCA(n_components=30, random_state=42)
X_pca= pca.fit_transform(X_sc)

ev  = pca.explained_variance_ratio_
n95 = int(np.argmax(np.cumsum(ev) >= 0.95)) + 1

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Scree
ax = axes[0]
ax.bar(range(1, len(ev)+1), ev*100, color="#3B7DBE", edgecolor="white")
ax.set_xlabel("Principal component")
ax.set_ylabel("Variance explained (%)")
ax.set_title("Scree plot")

# Cumulative
ax = axes[1]
ax.plot(range(1, len(ev)+1), np.cumsum(ev)*100, "o-",
        ms=4, color="#3B7DBE")
ax.axhline(95, color="red", lw=0.8, linestyle="--", label=f"95% ({n95} PCs)")
ax.axhline(99, color="orange", lw=0.8, linestyle="--", label="99%")
ax.set_xlabel("Number of components")
ax.set_ylabel("Cumulative variance (%)")
ax.set_title("Cumulative explained variance")
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

print(f"PC1 explains : {ev[0]*100:.1f}%")
print(f"PC1+PC2      : {sum(ev[:2])*100:.1f}%")
print(f"95% threshold: {n95} components")
print(f"99% threshold: {int(np.argmax(np.cumsum(ev) >= 0.99))+1} components")

## 14. PCA Biplots — Type, Latitude, Site

In [ ]:
fi_mask  = df[TARGET_COL].notna().values
y_types  = df[TARGET_COL].values  # includes NaN for Estonian plots
lats     = df[LAT_COL].values

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# ── Panel A: by peatland type ──
ax = axes[0]
ax.scatter(X_pca[~fi_mask, 0], X_pca[~fi_mask, 1],
           s=10, alpha=0.35, color="lightgray", label="Estonia")
for t in types:
    idx = fi_mask & (df[TARGET_COL] == t).values
    ax.scatter(X_pca[idx, 0], X_pca[idx, 1], s=18, alpha=0.8,
               color=t2c[t], label=t)
ax.set_xlabel(f"PC1 ({ev[0]*100:.1f}%)")
ax.set_ylabel(f"PC2 ({ev[1]*100:.1f}%)")
ax.set_title("A: Coloured by peatland type")
ax.legend(fontsize=6, ncol=2, loc="best")

# ── Panel B: by latitude ──
ax = axes[1]
sc = ax.scatter(X_pca[:, 0], X_pca[:, 1],
                c=lats, cmap="RdYlGn_r", s=14, alpha=0.75,
                norm=Normalize(lats.min(), lats.max()))
plt.colorbar(sc, ax=ax, label="Latitude (°N)")
ax.set_xlabel(f"PC1 ({ev[0]*100:.1f}%)")
ax.set_ylabel(f"PC2 ({ev[1]*100:.1f}%)")
ax.set_title("B: Coloured by latitude")

# ── Panel C: by site ──
ax = axes[2]
site_codes = df[SITE_COL].astype("category").cat.codes
sc2 = ax.scatter(X_pca[:, 0], X_pca[:, 1],
                 c=site_codes, cmap="tab20", s=14, alpha=0.75)
# Add site labels at centroid
for i, site in enumerate(sorted(df[SITE_COL].unique())):
    idx = (df[SITE_COL] == site).values
    cx, cy = X_pca[idx, 0].mean(), X_pca[idx, 1].mean()
    ax.text(cx, cy, site, fontsize=5.5, ha="center", va="center",
            fontweight="bold", color="black")
ax.set_xlabel(f"PC1 ({ev[0]*100:.1f}%)")
ax.set_ylabel(f"PC2 ({ev[1]*100:.1f}%)")
ax.set_title("C: Coloured by site (labels = centroids)")

plt.suptitle("PCA of 2151 hyperspectral bands — 446 plots", fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# PC3 vs PC4 — additional structure
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for ax, (pc_a, pc_b) in zip(axes, [(2, 3), (0, 2)]):
    ax.scatter(X_pca[~fi_mask, pc_a], X_pca[~fi_mask, pc_b],
               s=10, alpha=0.35, color="lightgray")
    for t in types:
        idx = fi_mask & (df[TARGET_COL] == t).values
        ax.scatter(X_pca[idx, pc_a], X_pca[idx, pc_b],
                   s=16, alpha=0.8, color=t2c[t], label=t)
    ax.set_xlabel(f"PC{pc_a+1} ({ev[pc_a]*100:.1f}%)")
    ax.set_ylabel(f"PC{pc_b+1} ({ev[pc_b]*100:.1f}%)")
    ax.set_title(f"PC{pc_a+1} vs PC{pc_b+1}")

axes[0].legend(fontsize=6, ncol=2)
plt.suptitle("Additional PCA projections — by peatland type", fontsize=10, y=1.01)
plt.tight_layout()
plt.show()

## 15. PCA Loading Vectors

In [ ]:
n_load = 5  # show first 5 loading vectors

fig, axes = plt.subplots(n_load, 1, figsize=(14, 3 * n_load), sharex=True)
region_colors = {"VIS": "#FFFDE7", "NIR": "#E8F5E9", "SWIR-1": "#FFF3E0", "SWIR-2": "#FCE4EC"}
region_spans  = [(350, 700, "VIS"), (700, 1330, "NIR"),
                 (1550, 1760, "SWIR-1"), (2025, 2310, "SWIR-2")]

for i, ax in enumerate(axes):
    load = pca.components_[i]
    # Shade regions
    for (a, b, lbl) in region_spans:
        mask = (wls >= a) & (wls <= b)
        if mask.sum() > 0:
            ax.axvspan(a, b, color=region_colors.get(lbl, "white"), alpha=0.3, zorder=0)
    ax.fill_between(wls, 0, load, where=(load > 0), color="#3B7DBE", alpha=0.5, lw=0)
    ax.fill_between(wls, 0, load, where=(load < 0), color="#E07B39", alpha=0.5, lw=0)
    ax.plot(wls, load, color="#333", lw=0.7)
    ax.axhline(0, color="k", lw=0.4)
    ax.set_ylabel(f"PC{i+1} loading\n({ev[i]*100:.1f}%)", fontsize=8)

# Add region labels on top axis
for (a, b, lbl) in region_spans:
    axes[0].text((a+b)//2, axes[0].get_ylim()[1]*1.02, lbl,
                 ha="center", fontsize=8, color="#555")

axes[-1].set_xlabel("Wavelength (nm)")
axes[0].set_title(f"PCA loading vectors (PC1–{n_load})\nBlue = positive loading, orange = negative")
plt.tight_layout()
plt.show()

## 16. Spectral Indices

In [ ]:
def get_band(nm):
    """Get the reflectance array for a specific wavelength."""
    col = f"wl{nm}"
    if col in df.columns:
        return df[col].astype(float).values
    # Nearest available
    avail = wl_array(get_spec_cols(False))
    nearest = avail[np.argmin(np.abs(avail - nm))]
    return df[f"wl{nearest}"].astype(float).values

R660  = get_band(660)
R800  = get_band(800)
R860  = get_band(860)
R1240 = get_band(1240)
R2100 = get_band(2100)
R2200 = get_band(2200)

# Indices
NDVI   = (R800  - R660)  / (R800  + R660  + 1e-9)   # greenness
NDWI   = (R860  - R1240) / (R860  + R1240 + 1e-9)   # water content
CAI    = 0.5 * (R2000_approx := get_band(2000)) + get_band(2200) - get_band(2100)  # char/litter
# Simpler CAI: 0.5*(R2000+R2200) - R2100
CAI    = 0.5 * (get_band(2000) + R2200) - R2100

idx_df = df[[SITE_COL, COUNTRY_COL]].copy()
idx_df[TARGET_COL] = df[TARGET_COL]
idx_df["NDVI"] = NDVI
idx_df["NDWI"] = NDWI
idx_df["CAI"]  = CAI

print("Global index statistics:")
print(idx_df[["NDVI","NDWI","CAI"]].describe().round(4))

In [ ]:
# Boxplots of indices by peatland type
fi_idx = idx_df.dropna(subset=[TARGET_COL])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (index_name, idx_label) in zip(axes, [
        ("NDVI", "NDVI (greenness)"),
        ("NDWI", "NDWI (water content)"),
        ("CAI",  "CAI (cellulose absorption)")]):
    data   = [fi_idx[fi_idx[TARGET_COL] == t][index_name].dropna() for t in types]
    bp = ax.boxplot(data, patch_artist=True, medianprops={"color": "k", "lw": 1.5})
    for patch, t in zip(bp["boxes"], types):
        patch.set_facecolor(t2c[t])
    ax.set_xticks(range(1, len(types)+1))
    ax.set_xticklabels(types, rotation=35, ha="right", fontsize=7)
    ax.set_ylabel(idx_label)
    ax.set_title(idx_label)

plt.suptitle("Spectral indices by peatland type", fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# NDVI vs. NDWI scatter — coloured by type
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(fi_idx["NDWI"], fi_idx["NDVI"],
           c=[t2c[t] for t in fi_idx[TARGET_COL]], s=20, alpha=0.7)
patches = [mpatches.Patch(color=t2c[t], label=t) for t in types]
ax.legend(handles=patches, fontsize=7, ncol=2, loc="lower right")
ax.set_xlabel("NDWI (water content)")
ax.set_ylabel("NDVI (greenness)")
ax.set_title("NDVI vs. NDWI scatter — coloured by peatland type")
plt.tight_layout()
plt.show()

## 17. SWIR Quality Class Audit

In [ ]:
if "SWIR_class" in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # SWIR class by peatland type
    ax = axes[0]
    cross_swir = fi.groupby([TARGET_COL, "SWIR_class"]).size().unstack(fill_value=0)
    cross_swir_pct = cross_swir.div(cross_swir.sum(axis=1), axis=0)
    cross_swir_pct.plot(kind="bar", stacked=True, ax=ax,
                        color=["#2E7D32", "#FBC02D", "#C62828"][:len(cross_swir_pct.columns)],
                        edgecolor="white", linewidth=0.4)
    ax.set_ylabel("Proportion")
    ax.set_title("SWIR quality class by peatland type")
    ax.legend(title="SWIR class", fontsize=8)
    plt.setp(ax.get_xticklabels(), rotation=35, ha="right", fontsize=7)

    # SWIR class vs NIR reflectance
    ax = axes[1]
    for cls in sorted(df["SWIR_class"].dropna().unique()):
        sub = df[df["SWIR_class"] == cls]
        swir_vals = sub["wl2100"].astype(float) if "wl2100" in df.columns else sub["wl1600"].astype(float)
        ax.hist(swir_vals.dropna(), bins=25, alpha=0.5, label=f"Class {cls}")
    ax.set_xlabel("Reflectance")
    ax.set_ylabel("Count")
    ax.set_title("SWIR reflectance distribution by quality class")
    ax.legend(fontsize=8)

    plt.tight_layout()
    plt.show()
    
    print("\nSWIR class × peatland type count table:")
    print(cross_swir.to_string())

## 18. Summary & Key Findings

In [ ]:
print("=" * 65)
print("EDA SUMMARY — KEY FINDINGS")
print("=" * 65)

print(f"""
1. DATASET STRUCTURE
   • 446 plots across 13 sites in Finland (n={n_fi}) and Estonia (n={n_ee})
   • Latitude range: {df[LAT_COL].min():.2f}–{df[LAT_COL].max():.2f} °N 
     (~{(df[LAT_COL].max()-df[LAT_COL].min())*111:.0f} km north–south gradient)
   • 2151 raw bands; {len(spec_cols_clean)} usable bands after excluding
     water-absorption regions (1330–1549, 1761–2024, 2311–2500 nm)

2. CLASS DISTRIBUTION
   • {len(types)} Finnish peatland types; class imbalance ratio 
     {counts.max()}/{counts.min()} = {counts.max()//counts.min()}×
   • Rare types may need merging or class-weighted classifiers

3. SPECTRAL CHARACTERISTICS
   • PC1 alone explains {ev[0]*100:.1f}% of spectral variance — 
     the data has very high intrinsic redundancy
   • 95% variance captured by {n95} PCA components
   • Red-edge (700–800 nm) shows clear separation between types 
     in mean spectra — vegetation density signal
   • High between-plot variance in VIS and red-edge; 
     NIR plateau is more stable

4. GEOGRAPHIC GRADIENT
   • PC1 aligns with latitude — the north–south climate gradient 
     is encoded in the spectra
   • *** IMPORTANT FOR MODELLING ***: this means classifiers trained 
     with random splits will appear better than they are; 
     use leave-one-site-out cross-validation

5. PFT PATTERNS
   • Sphagnum mosses dominate in bogs; graminoids in fens
   • Sphagnum and graminoids show strongly contrasting 
     spectral correlations — key discriminators
   • Lichens show high NIR reflectance; water plots have 
     very low reflectance across all bands

6. SPECTRAL INDICES
   • NDVI separates forested from open peatlands well
   • NDWI correlates with wet/dry gradient (bog vs fen)
   • CAI sensitive to litter/dead plant material

7. MODELLING RECOMMENDATIONS (next steps)
   • Feature space: use clean bands (exclude noisy SWIR) OR 
     reduce to top PCA components first
   • Baseline: PLS-DA (matches original paper approach)
   • Strong baseline: Random Forest (handles class imbalance, 
     provides band importance)
   • CV strategy: GroupKFold with Site as group — mandatory
   • Additional targets: predict latitude (regression) to test 
     whether spectra encode geographic location
""")